# Minimal Decoding From Checkpoint

Load a trained checkpoint and sample text with temperature.
Supports byte-level decoding by default, and optional BPE decoding.


In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

# Ensure repo-root imports work from notebook.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.transformer.transformer_lm import TransformerLM
from src.tokenizer.tokenizer import Tokenizer


In [ ]:
# --- User config ---
checkpoint_path = REPO_ROOT / "artifacts/checkpoints/minimal_ckpt.pt"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# Must match training hyperparameters.
vocab_size = 256
context_length = 128
d_model = 320
num_heads = 8
d_ff = 1280
num_layers = 6
theta = 10000.0

# Optional BPE decoding (set both paths to use tokenizer.decode).
vocab_pkl = None  # e.g., REPO_ROOT / "artifacts/tokenized/tinystories_vocab.pkl"
merges_pkl = None # e.g., REPO_ROOT / "artifacts/tokenized/tinystories_merges.pkl"
special_tokens = ["<|endoftext|>"]


In [ ]:
def build_model():
    model = TransformerLM(
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        vocab_size=vocab_size,
        context_length=context_length,
        num_layers=num_layers,
        theta=theta,
        device=device,
    ).to(device)
    model.eval()
    return model

model = build_model()
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Iteration: {ckpt.get('iteration', 'n/a')}")


In [ ]:
tokenizer = None
if vocab_pkl is not None and merges_pkl is not None:
    tokenizer = Tokenizer.from_files(str(vocab_pkl), str(merges_pkl), special_tokens=special_tokens)
    print("Using BPE tokenizer decode")
else:
    print("Using byte-level decode")


def encode_prompt(prompt: str):
    if tokenizer is not None:
        return tokenizer.encode(prompt)
    return list(prompt.encode("utf-8"))


def decode_ids(ids):
    if tokenizer is not None:
        return tokenizer.decode(ids)
    return bytes(ids).decode("utf-8", errors="replace")


In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 1.0):
    assert temperature > 0, "temperature must be > 0"

    ids = encode_prompt(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -context_length:]
        logits = model(x_cond)  # model returns logits
        next_logits = logits[:, -1, :] / temperature
        next_probs = F.softmax(next_logits, dim=-1)

        next_id = torch.multinomial(next_probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return decode_ids(x[0].tolist())


In [ ]:
prompt = "Once upon a time there was"
out = generate(prompt=prompt, max_new_tokens=20, temperature=0.9)
print(out)
